In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [3]:
df_master = pd.read_pickle('master_eda.pkl')
print(f"Kích thước bảng Master ban đầu: {df_master.shape}")

Kích thước bảng Master ban đầu: (59707, 23)


In [4]:
df_master.head()

,msno,active_days,total_secs,num_100,num_25,num_unq,secs_first_half,secs_second_half,city,age,...,total_paid,auto_renew_ratio,cancel_count,is_churn,snapshot_id,churn_label,has_auto_renew,has_cancel,decay_ratio,skip_ratio
0,//+GDngYumw0H8Z75Mpega0HOmNqtDQJ9USLYllbIlI=,16,125361.195312,480.0,96.0,546.0,52761.195312,72600.000000,22.0,25.0,...,0.0,127.0,0.0,0,snap_1_jan_feb,Non-Churn (0),Auto-Renew,No Cancel,0.376004,0.166378
1,//DTI+atVkSg9WADqkkwXiLHtMIzsatHSqam1RYmQd8=,29,309839.000000,1016.0,299.0,1190.0,157372.343750,152466.640625,13.0,21.0,...,1.0,127.0,0.0,0,snap_1_jan_feb,Non-Churn (0),Auto-Renew,No Cancel,-0.031172,0.227204
2,//ow4l7MG9mmPyeugoSabvIgk1WlSpTFMqF4sOe+JNw=,2,887.052979,3.0,2.0,5.0,887.052979,0.000000,1.0,-1.0,...,1.0,127.0,0.0,0,snap_1_jan_feb,Non-Churn (0),Auto-Renew,No Cancel,-0.998874,0.333333
3,//PSDRqlc3I2P5vSUkigd4Sef+DsevS3yIpf7NK+HpI=,3,2084.226074,7.0,5.0,13.0,0.000000,2084.226074,1.0,-1.0,...,0.0,0.0,0.0,1,snap_1_jan_feb,Churn (1),No Auto-Renew,No Cancel,2084.226074,0.384615
4,/+/mWTmR9QC+5rt67nIM/6R9DCWvmNfLd/ePIzL3LNk=,1,822.215027,2.0,1.0,6.0,0.000000,822.215027,1.0,-1.0,...,0.0,0.0,0.0,1,snap_1_jan_feb,Churn (1),No Auto-Renew,No Cancel,822.215027,0.250000


# FEATURE ENGINEERING

In [6]:
df_master['decay_ratio'] = (df_master['secs_second_half'] - df_master['secs_first_half']) / (df_master['secs_first_half'] + 1.0)
df_master['decay_ratio'] = df_master['decay_ratio'].clip(lower=-1.0, upper=10.0)

df_master['skip_ratio'] = df_master['num_25'] / (df_master['num_100'] + df_master['num_25'] + 1.0)
df_master['avg_secs_per_active_day'] = df_master['total_secs'] / (df_master['active_days'] + 1e-5)

df_master.shape

(59707, 24)

In [7]:
skewed_cols = [
    'total_secs', 'num_100', 'num_25', 'num_unq', 
    'total_paid', 'avg_secs_per_active_day'
]
print(f" Áp dụng np.log1p() cho các biến phân phối lệch phải: {skewed_cols}")
for col in skewed_cols:
    df_master[col] = np.log1p(np.maximum(0, df_master[col]))

df_encoded = pd.get_dummies(df_master, columns=['gender'], drop_first=False, dtype=int)

 Áp dụng np.log1p() cho các biến phân phối lệch phải: ['total_secs', 'num_100', 'num_25', 'num_unq', 'total_paid', 'avg_secs_per_active_day']


# TRAIN/VAL/TEST BẰNG TEMPORAL SPLIT

In [9]:
cols_to_drop = [
    'msno', 'is_churn', 'snapshot_id', 
    'churn_label', 'has_auto_renew', 'has_cancel',
    'secs_first_half', 'secs_second_half'
]
existing_drop_cols = [c for c in cols_to_drop if c in df_encoded.columns]

snap1_mask = df_encoded['snapshot_id'].str.strip() == 'snap_1_jan_feb'
snap2_mask = df_encoded['snapshot_id'].str.strip() == 'snap_2_feb_mar'

df_train_val = df_encoded[snap1_mask].copy()

train_users = set(df_train_val['msno'].unique())
df_test = df_encoded[snap2_mask & (~df_encoded['msno'].isin(train_users))].copy()

X_train_val = df_train_val.drop(columns=existing_drop_cols)
y_train_val = df_train_val['is_churn'].astype(int)

X_test = df_test.drop(columns=existing_drop_cols)
y_test = df_test['is_churn'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, random_state=42, stratify=y_train_val
)

print(f" -> Kích thước X_train : {X_train.shape}")
print(f" -> Kích thước X_val   : {X_val.shape}")
print(f" -> Kích thước X_test  : {X_test.shape}")

 -> Kích thước X_train : (25643, 18)
 -> Kích thước X_val   : (4526, 18)
 -> Kích thước X_test  : (4401, 18)


# XỬ LÝ OUTLIER, SCALING VÀ SMOTE

In [11]:
valid_age_mask = (X_train['age'] > 0) & (X_train['age'] <= 100)
valid_age_median = float(X_train.loc[valid_age_mask, 'age'].median())

for df_sub in [X_train, X_val, X_test]:
    df_sub['is_age_missing'] = ((df_sub['age'] <= 0) | (df_sub['age'] > 100)).astype(int)
    df_sub['age'] = df_sub['age'].apply(
        lambda x: valid_age_median if (x <= 0 or x > 100) else x
    )
feature_names = X_train.columns.tolist()

In [12]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val), columns=feature_names, index=X_val.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=feature_names, index=X_test.index)

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print(f"4. [SMOTE] Đã cân bằng lớp thiểu số trên tập Train:")
print(f"   - Trước SMOTE: {dict(y_train.value_counts())}")
print(f"   - Sau SMOTE  : {dict(y_train_res.value_counts())}")

4. [SMOTE] Đã cân bằng lớp thiểu số trên tập Train:
   - Trước SMOTE: {0: np.int64(21366), 1: np.int64(4277)}
   - Sau SMOTE  : {0: np.int64(21366), 1: np.int64(21366)}
=> HOÀN TẤT PREPROCESSING THEO ĐÚNG THỨ TỰ CHUẨN KHOA HỌC DỮ LIỆU!


In [14]:
import os
import joblib

os.makedirs("artifacts", exist_ok=True)

dataset_artifacts = {
    'X_train_res': X_train_res,
    'y_train_res': y_train_res,
    'X_val_scaled': X_val_scaled,
    'y_val': y_val,
    'X_test_scaled': X_test_scaled,
    'y_test': y_test,
    'feature_names': feature_names
}

joblib.dump(dataset_artifacts, "artifacts/processed_data.pkl")
print("-> Đã lưu toàn bộ dữ liệu Train, Val, Test vào artifacts/processed_data.pkl")

-> Đã lưu toàn bộ dữ liệu Train, Val, Test vào artifacts/processed_data.pkl
